# boolean-mask-combine — worked example 2: Keep finite, positive, non-flagged entries

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `boolean-mask-combine`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common data-cleaning pattern is to keep only entries that satisfy several conditions at once: finite, strictly positive, and not externally flagged. You build one bool tensor per condition, then combine with `&` (and use `~` to negate the flag). `~t.isfinite(x)` flips a finite-mask into a non-finite mask; equivalently `t.isfinite(x)` is already the keep-condition for finiteness.

## Worked solution

**Goal:** given a `(N,)` tensor `x` (possibly containing inf/nan) and a `(N,)` bool tensor `flagged`, return a `(N,)` bool mask that is `True` where the entry is finite AND strictly positive AND not flagged.

**Step 1 — finiteness predicate.** `t.isfinite(x)` returns `True` for normal floats and `False` for `inf`/`nan`. This is already in 'keep' polarity, so no negation needed.

**Step 2 — positivity predicate.** `(x > 0)` is a `(N,)` bool tensor. Note that for `nan`, `nan > 0` is `False`, so positivity already excludes nan, but combining with the explicit finiteness check makes intent clear and also rejects `+inf`.

**Step 3 — un-flag predicate.** `flagged` marks rows to drop, so the keep-condition is its negation: `~flagged`. `~` is elementwise logical-not on a bool tensor.

**Step 4 — AND all three.** `t.isfinite(x) & (x > 0) & (~flagged)`. All three are `(N,)` bool tensors, so elementwise `&` yields the final `(N,)` bool keep-mask. We use AND because we keep an entry only when every condition passes simultaneously.

**Why it works:** the valid set is the intersection of three predicates; intersection is elementwise conjunction. Wrapping `x > 0` and `~flagged` in parentheses guards against `&` binding tighter than `>`.

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat
np.random.seed(0)
t.manual_seed(0)

def keep_valid(x: Tensor, flagged: Tensor) -> Tensor:
    return t.isfinite(x) & (x > 0) & (~flagged)

x = t.tensor([1.5, -2.0, float('inf'), float('nan'), 3.0, 0.0])
flagged = t.tensor([False, False, False, False, True, False])
mask = keep_valid(x, flagged)
print(mask)
print(mask.dtype, mask.shape)